# 第6章: モデル評価とハイパーパラメータチューニングの実践

この Notebook は、原本 `machine-learning-book/ch06/ch06.ipynb` を最新の Python / scikit-learn 環境で継続検証しやすい形に移行したものです。  
パイプライン、交差検証、学習曲線、ハイパーパラメータ探索、評価指標、不均衡データへの対処という章の主要テーマを、CI 上でヘッドレス実行できる構成で再現します。


## この Notebook で確認すること

- 現在の `uv` 環境で Python と主要パッケージのバージョンを確認する。
- 読み取り専用サブモジュール `machine-learning-book/` から図版とデータを参照できることを確認する。
- Breast Cancer Wisconsin データセットでパイプライン、交差検証、学習曲線、検証曲線を再現する。
- Grid Search、Randomized Search、Successive Halving、ネストした交差検証の代表例を最新 API で確認する。
- 混同行列、適合率、再現率、ROC 曲線、不均衡データへのアップサンプリングを CI で止まらない形で実行する。


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    auc,
    confusion_matrix,
    f1_score,
    make_scorer,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    HalvingRandomSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    learning_curve,
    train_test_split,
    validation_curve,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import resample


In [ ]:
PACKAGE_NAMES = [
    "ipykernel",
    "matplotlib",
    "nbmake",
    "numpy",
    "pandas",
    "pytest",
    "scikit-learn",
]


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "machine-learning-book").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("リポジトリルートを特定できませんでした。")


repo_root = find_repo_root(Path.cwd())
chapter_root = repo_root / "machine-learning-book" / "ch06"
figure_dir = chapter_root / "figures"
data_path = chapter_root / "wdbc.data"

if not data_path.exists():
    raise FileNotFoundError(f"データファイルが見つかりません: {data_path}")

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Matplotlib backend: {matplotlib.get_backend()}")
print(f"Chapter root: {chapter_root}")


In [ ]:
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=["パッケージ", "バージョン"],
)
package_versions


## 原本図版の参照

移行版 Notebook は `src/ch06/` に置きつつ、書籍の図版アセットは読み取り専用の `machine-learning-book/ch06/figures/` から参照します。  
ここでは章の要点に対応する図版をいくつか表示し、`src/` 側から相対パスに依存せずアクセスできることを確認します。


In [ ]:
selected_figures = [
    ("06_01.png", 440),
    ("06_05.png", 440),
    ("06_09.png", 320),
]

for name, width in selected_figures:
    figure_path = figure_dir / name
    if not figure_path.exists():
        raise FileNotFoundError(f"図版が見つかりません: {figure_path}")
    display(Image(filename=str(figure_path), width=width))


## Breast Cancer Wisconsin データセットの準備

原本では UCI リポジトリからの読み込みが含まれていましたが、CI での安定性を優先して、この移行版ではサブモジュール内に同梱されている `wdbc.data` を使います。  
ラベルは原本どおり `M` を 1、`B` を 0 にエンコードし、層化付きで訓練用とテスト用に分割します。


In [ ]:
feature_names = [
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",
    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave points_se",
    "symmetry_se",
    "fractal_dimension_se",
    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave points_worst",
    "symmetry_worst",
    "fractal_dimension_worst",
]
columns = ["id", "diagnosis", *feature_names]

df = pd.read_csv(data_path, header=None, names=columns)
X = df.loc[:, feature_names].to_numpy()
y_text = df["diagnosis"].to_numpy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=1,
)

pd.Series(
    {
        "全サンプル数": len(df),
        "特徴量数": X.shape[1],
        "訓練サンプル数": len(X_train),
        "テストサンプル数": len(X_test),
        "ラベル対応": label_mapping,
    }
)


## パイプラインと交差検証

標準化、PCA、ロジスティック回帰を 1 本のパイプラインにまとめ、ホールドアウト評価と層化 k 分割交差検証を比較します。  
原本は 10-fold を使っていましたが、CI での実行時間を抑えるため、移行版では 5-fold に調整しています。


In [ ]:
pipe_lr = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    LogisticRegression(max_iter=1000, solver="liblinear", random_state=1),
)

pipe_lr.fit(X_train, y_train)
test_accuracy = pipe_lr.score(X_test, y_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
fold_scores = []
for fold_index, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train), start=1):
    estimator = clone(pipe_lr)
    estimator.fit(X_train[train_idx], y_train[train_idx])
    fold_scores.append(
        {
            "fold": fold_index,
            "train_class_counts": np.bincount(y_train[train_idx]).tolist(),
            "valid_accuracy": round(estimator.score(X_train[valid_idx], y_train[valid_idx]), 4),
        }
    )

cv_scores = cross_val_score(pipe_lr, X_train, y_train, cv=cv, n_jobs=1)

print(f"テスト正解率: {test_accuracy:.3f}")
print(f"5-fold CV 平均: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
pd.DataFrame(fold_scores)


## 学習曲線と検証曲線

学習曲線ではデータ量に対する訓練精度と検証精度の推移を見て、検証曲線では正則化係数 `C` に対する過学習・過少学習の傾向を確認します。  
描画は `MPLBACKEND=Agg` 前提でも停止しないように通常の `plt.show()` だけを使います。


In [ ]:
curve_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, solver="liblinear", random_state=1),
)

train_sizes, train_scores, valid_scores = learning_curve(
    estimator=curve_pipeline,
    X=X_train,
    y=y_train,
    train_sizes=np.linspace(0.2, 1.0, 5),
    cv=cv,
    n_jobs=1,
)

param_range = np.logspace(-3, 2, 6)
val_train_scores, val_valid_scores = validation_curve(
    estimator=curve_pipeline,
    X=X_train,
    y=y_train,
    param_name="logisticregression__C",
    param_range=param_range,
    cv=cv,
    n_jobs=1,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Train")
axes[0].fill_between(
    train_sizes,
    train_scores.mean(axis=1) - train_scores.std(axis=1),
    train_scores.mean(axis=1) + train_scores.std(axis=1),
    alpha=0.15,
)
axes[0].plot(train_sizes, valid_scores.mean(axis=1), marker="s", linestyle="--", label="Validation")
axes[0].fill_between(
    train_sizes,
    valid_scores.mean(axis=1) - valid_scores.std(axis=1),
    valid_scores.mean(axis=1) + valid_scores.std(axis=1),
    alpha=0.15,
)
axes[0].set_title("Learning Curve")
axes[0].set_xlabel("Number of training examples")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0.85, 1.01)
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="lower right")

axes[1].plot(param_range, val_train_scores.mean(axis=1), marker="o", label="Train")
axes[1].plot(param_range, val_valid_scores.mean(axis=1), marker="s", linestyle="--", label="Validation")
axes[1].fill_between(
    param_range,
    val_train_scores.mean(axis=1) - val_train_scores.std(axis=1),
    val_train_scores.mean(axis=1) + val_train_scores.std(axis=1),
    alpha=0.15,
)
axes[1].fill_between(
    param_range,
    val_valid_scores.mean(axis=1) - val_valid_scores.std(axis=1),
    val_valid_scores.mean(axis=1) + val_valid_scores.std(axis=1),
    alpha=0.15,
)
axes[1].set_xscale("log")
axes[1].set_title("Validation Curve")
axes[1].set_xlabel("C")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.85, 1.01)
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


## ハイパーパラメータ探索とモデル選択

SVC パイプラインに対して Grid Search、Randomized Search、Successive Halving を比較し、その後にネストした交差検証で SVC と決定木を比べます。  
原本の意図は維持しつつ、探索空間と分割数は CI 向けに現実的なサイズへ縮小しています。


In [ ]:
svc_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("svc", SVC(random_state=1)),
    ]
)

search_space = [
    {"svc__kernel": ["linear"], "svc__C": [0.1, 1.0, 10.0]},
    {"svc__kernel": ["rbf"], "svc__C": [0.1, 1.0, 10.0], "svc__gamma": [0.01, 0.1, 1.0]},
]

grid_search = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=search_space,
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    refit=True,
)
grid_search.fit(X_train, y_train)

random_search = RandomizedSearchCV(
    estimator=svc_pipeline,
    param_distributions=search_space,
    n_iter=6,
    scoring="accuracy",
    cv=3,
    n_jobs=1,
    random_state=1,
    refit=True,
)
random_search.fit(X_train, y_train)

halving_search = HalvingRandomSearchCV(
    estimator=svc_pipeline,
    param_distributions=search_space,
    n_candidates=6,
    factor=2,
    cv=3,
    n_jobs=1,
    random_state=1,
    scoring="accuracy",
)
halving_search.fit(X_train, y_train)

search_summary = pd.DataFrame(
    [
        {
            "探索手法": "GridSearchCV",
            "CV平均": round(grid_search.best_score_, 4),
            "テスト正解率": round(grid_search.best_estimator_.score(X_test, y_test), 4),
            "最良パラメータ": str(grid_search.best_params_),
        },
        {
            "探索手法": "RandomizedSearchCV",
            "CV平均": round(random_search.best_score_, 4),
            "テスト正解率": round(random_search.best_estimator_.score(X_test, y_test), 4),
            "最良パラメータ": str(random_search.best_params_),
        },
        {
            "探索手法": "HalvingRandomSearchCV",
            "CV平均": round(halving_search.best_score_, 4),
            "テスト正解率": round(halving_search.best_estimator_.score(X_test, y_test), 4),
            "最良パラメータ": str(halving_search.best_params_),
        },
    ]
)

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=2)

nested_svc = cross_val_score(
    GridSearchCV(svc_pipeline, search_space, scoring="accuracy", cv=inner_cv, n_jobs=1),
    X_train,
    y_train,
    scoring="accuracy",
    cv=outer_cv,
    n_jobs=1,
)

nested_tree = cross_val_score(
    GridSearchCV(
        DecisionTreeClassifier(random_state=1),
        param_grid={"max_depth": [1, 2, 3, 4, None]},
        scoring="accuracy",
        cv=inner_cv,
        n_jobs=1,
    ),
    X_train,
    y_train,
    scoring="accuracy",
    cv=outer_cv,
    n_jobs=1,
)

display(search_summary)
pd.DataFrame(
    [
        {"モデル": "SVC", "nested_cv_mean": nested_svc.mean(), "nested_cv_std": nested_svc.std()},
        {"モデル": "DecisionTree", "nested_cv_mean": nested_tree.mean(), "nested_cv_std": nested_tree.std()},
    ]
)


## 混同行列と評価指標

最良の SVC モデルを使って混同行列を可視化し、適合率、再現率、F1、MCC をまとめて確認します。  
さらに、陽性クラスを切り替えた `F1` を最適化する scorer を作り、評価指標の設計が探索結果に影響することも確認します。


In [ ]:
best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test)
confmat = confusion_matrix(y_test, y_pred, labels=[1, 0])

fig, ax = plt.subplots(figsize=(3.2, 3.2))
ax.matshow(confmat, cmap=plt.cm.Blues, alpha=0.3)
for i in range(confmat.shape[0]):
    for j in range(confmat.shape[1]):
        ax.text(j, i, confmat[i, j], va="center", ha="center")
ax.set_xticks([0, 1], labels=["Pred: malignant", "Pred: benign"])
ax.set_yticks([0, 1], labels=["True: malignant", "True: benign"])
plt.tight_layout()
plt.show()

metric_summary = pd.Series(
    {
        "precision": round(precision_score(y_test, y_pred), 4),
        "recall": round(recall_score(y_test, y_pred), 4),
        "f1": round(f1_score(y_test, y_pred), 4),
        "mcc": round(matthews_corrcoef(y_test, y_pred), 4),
    }
)

minority_f1_scorer = make_scorer(f1_score, pos_label=0)
metric_grid = [
    {"svc__kernel": ["linear"], "svc__C": [0.1, 1.0, 10.0]},
    {"svc__kernel": ["rbf"], "svc__C": [0.1, 1.0, 10.0], "svc__gamma": [0.01, 0.1, 1.0]},
]
minority_grid = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=metric_grid,
    scoring=minority_f1_scorer,
    cv=3,
    n_jobs=1,
)
minority_grid.fit(X_train, y_train)

display(metric_summary.to_frame("値"))
pd.Series(
    {
        "少数側F1最適化の最良スコア": round(minority_grid.best_score_, 4),
        "少数側F1最適化の最良パラメータ": str(minority_grid.best_params_),
    }
)


## ROC 曲線

原本と同様に 2 つの特徴量だけを使ったロジスティック回帰で、各 fold の ROC 曲線と平均 ROC 曲線を描きます。  
`numpy.interp` を用いて平均 TPR を計算し、最新環境でそのまま動くようにしています。


In [ ]:
roc_pipeline = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    LogisticRegression(C=100.0, solver="liblinear", random_state=1),
)
X_train_two_features = X_train[:, [4, 14]]
roc_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)

mean_fpr = np.linspace(0, 1, 100)
mean_tpr = np.zeros_like(mean_fpr)

fig, ax = plt.subplots(figsize=(6, 4.5))
for fold_index, (train_idx, valid_idx) in enumerate(roc_cv.split(X_train_two_features, y_train), start=1):
    probas = roc_pipeline.fit(
        X_train_two_features[train_idx],
        y_train[train_idx],
    ).predict_proba(X_train_two_features[valid_idx])
    fpr, tpr, _ = roc_curve(y_train[valid_idx], probas[:, 1], pos_label=1)
    fold_auc = auc(fpr, tpr)
    mean_tpr += np.interp(mean_fpr, fpr, tpr)
    mean_tpr[0] = 0.0
    ax.plot(fpr, tpr, label=f"Fold {fold_index} (AUC={fold_auc:.2f})")

mean_tpr /= roc_cv.get_n_splits()
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
ax.plot(mean_fpr, mean_tpr, linestyle="--", color="black", linewidth=2, label=f"Mean ROC (AUC={mean_auc:.2f})")
ax.plot([0, 0, 1], [0, 1, 1], linestyle=":", color="black", label="Perfect performance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## クラス不均衡への対処

原本と同じ考え方で良性クラスを多数派、悪性クラスを少数派にした人工的な不均衡データを作り、  
多数派だけを予測するベースライン精度と、アップサンプリング後のクラス数を比較します。


In [ ]:
X_imb = np.vstack([X[y == 0], X[y == 1][:40]])
y_imb = np.hstack([y[y == 0], y[y == 1][:40]])

baseline_predictions = np.zeros_like(y_imb)
baseline_accuracy = (baseline_predictions == y_imb).mean()

X_upsampled, y_upsampled = resample(
    X_imb[y_imb == 1],
    y_imb[y_imb == 1],
    replace=True,
    n_samples=(y_imb == 0).sum(),
    random_state=123,
)

X_bal = np.vstack([X_imb[y_imb == 0], X_upsampled])
y_bal = np.hstack([y_imb[y_imb == 0], y_upsampled])

pd.DataFrame(
    [
        {"状態": "不均衡データ", "class_0": int((y_imb == 0).sum()), "class_1": int((y_imb == 1).sum())},
        {"状態": "アップサンプリング後", "class_0": int((y_bal == 0).sum()), "class_1": int((y_bal == 1).sum())},
    ]
)


In [ ]:
pd.Series(
    {
        "多数派だけを予測したときの精度": round(float(baseline_accuracy), 4),
        "不均衡データのサンプル数": len(y_imb),
        "バランス後のサンプル数": len(y_bal),
    }
)


## まとめ

この移行版では、原本 `ch06` の教育的意図を保ちながら、以下の点を最新環境向けに調整しました。

- UCI への外部アクセスをやめ、サブモジュール内の `wdbc.data` を使ってローカル完結にした。
- 交差検証の fold 数や探索回数を絞り、`pytest --nbmake` で現実的に通る実行時間へ調整した。
- 旧式のコード断片や typo を避け、現在の scikit-learn API でそのまま動く構成にした。
- 図版参照は `Path.cwd()` に固定せず、Notebook の実行場所が変わっても壊れにくい探索ロジックにした。
